# 🔥 OpenMythos Finance Model — Training

**Steps:** Runtime → Change runtime type → **T4 GPU**

Then run all cells (Ctrl+F9)

In [ ]:
#@title Step 1: Install Dependencies { display-mode: "form" }
!pip install -q transformers datasets peft bitsandbytes accelerate trl sentencepiece
!nvidia-smi
import torch
print(f'\n✅ PyTorch {torch.__version__}')
print(f'🎮 GPU: {torch.cuda.get_device_name(0)}')
print(f'💾 VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
#@title Step 2: Clone Repository { display-mode: "form" }
!git clone -b develop https://github.com/oyi77/OpenMythos-BerkahKarya.git
%cd OpenMythos-BerkahKarya
!wc -l data/finance/finance_dataset.jsonl
print('✅ Repo cloned, training data ready!')

In [ ]:
#@title Step 3: Configure Training { display-mode: "form" }
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" #@param ["TinyLlama/TinyLlama-1.1B-Chat-v1.0", "Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "meta-llama/Llama-3.2-1B-Instruct"] {allow-input: true}
LORA_R = 16 #@param {"type":"integer"}
LORA_ALPHA = 32 #@param {"type":"integer"}
BATCH_SIZE = 4 #@param {"type":"integer"}
GRAD_ACCUM = 4 #@param {"type":"integer"}
LEARNING_RATE = 2e-4 #@param {"type":"number"}
NUM_EPOCHS = 3 #@param {"type":"integer"}
MAX_SEQ_LENGTH = 2048 #@param {"type":"integer"}

print(f'📋 Config:')
print(f'  Model: {MODEL_ID}')
print(f'  LoRA: r={LORA_R}, alpha={LORA_ALPHA}')
print(f'  Batch: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}')
print(f'  LR: {LEARNING_RATE}')
print(f'  Epochs: {NUM_EPOCHS}')
print(f'  Max seq: {MAX_SEQ_LENGTH}')

In [ ]:
#@title Step 4: Load Data { display-mode: "form" }
import json
from datasets import Dataset

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

train_data = load_jsonl('data/finance/train.jsonl')
val_data = load_jsonl('data/finance/val.jsonl')

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f'✅ Train: {len(train_dataset):,} samples')
print(f'✅ Val: {len(val_dataset):,} samples')
print(f'📊 Sample: {train_data[0]["text"][:200]}...')

In [ ]:
#@title Step 5: Load Model { display-mode: "form" }
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f'📥 Loading {MODEL_ID}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f'\n✅ Model loaded on {model.device}')
print(f'💾 VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
#@title Step 6: Train! { display-mode: "form" }
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = "openmythos-finance-v1"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
)

print('🚀 Starting training...')
print(f'📊 {len(train_dataset)} train / {len(val_dataset)} val samples')
print(f'📊 {NUM_EPOCHS} epochs × {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)} steps/epoch')
print()

trainer.train()

print(f'\n✅ Training complete!')
print(f'📊 Best eval loss: {trainer.state.best_metric:.4f}')

In [ ]:
#@title Step 7: Save & Download { display-mode: "form" }
import json, os
from google.colab import files

# Save
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Metadata
meta = {
    "model_id": MODEL_ID,
    "train_samples": len(train_data),
    "val_samples": len(val_data),
    "epochs": NUM_EPOCHS,
    "lora_r": LORA_R,
    "best_eval_loss": trainer.state.best_metric,
}
with open(f"{OUTPUT_DIR}/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

# Zip & download
!zip -r {OUTPUT_DIR}.zip {OUTPUT_DIR}/
files.download(f'{OUTPUT_DIR}.zip')

print(f'\n✅ Model saved to {OUTPUT_DIR}/')
print(f'📊 Best eval loss: {trainer.state.best_metric:.4f}')
print(f'💾 Downloading {OUTPUT_DIR}.zip...')